In [ ]:
!pip install enoslib

In [1]:
!ssh rennes.grid5000.fr hostname

frennes


In [11]:
import logging
import os
import time
from grid5000 import Grid5000

conf_file = os.path.join(os.environ.get("HOME"), ".python-grid5000.yaml")
gk = Grid5000.from_yaml(conf_file)

In [12]:
JOB_NAME="test_fcquic_npf"
JOB_WALLTIME="0:15:00"
JOB_CLUSTER="gros"
NUM_CLIENT_NODES=1
NUM_SERVER_NODES=1

In [13]:
!scp ~/fcquic_applications_master_thesis/fcquic_chat/target/release/server nancy.g5k:~/bin/server
!scp ~/fcquic_applications_master_thesis/fcquic_chat/target/release/client nancy.g5k:~/bin/client

!scp ~/fcquic_applications_master_thesis/fcquic_chat/cert.crt nancy.g5k:~/cert.crt
!scp ~/fcquic_applications_master_thesis/fcquic_chat/cert.key nancy.g5k:~/cert.key


server                                        100% 6591KB  24.3MB/s   00:00    
client                                        100% 6247KB  30.0MB/s   00:00    
cert.crt                                      100% 1302    62.6KB/s   00:00    
cert.key                                      100% 1708    82.1KB/s   00:00    


In [14]:
import enoslib as en
from npf import enoslib as npf
from npf.output.transform.pandas import to_pandas

# from importlib import reload
# reload(npf)

conf = en.G5kConf.from_settings(job_name=JOB_NAME, walltime=JOB_WALLTIME).add_machine(
    roles=["client"], cluster=JOB_CLUSTER, nodes=NUM_CLIENT_NODES
).add_machine(
    roles=["server"], cluster=JOB_CLUSTER, nodes=NUM_SERVER_NODES
)  

# This will validate the configuration, but not reserve resources yet
provider = en.G5k(conf)

In [15]:
print("Reserving resources...")

# Get actual resources
roles, networks = provider.init()
display(roles)
display(networks)

Reserving resources...


/home/corentin/.local/lib/python3.12/site-packages/rich/live.py:221: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Finished 1 tasks (Granting root access on the nodes (sudo-g5k)) on 
{'gros-86.nancy.grid5000.fr', 'gros-83.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

{'client': {Host(address='gros-83.nancy.grid5000.fr', alias='gros-83.nancy.grid5000.fr', user='root', keyfile=None, port=None, extra={'gateway': 'access.grid5000.fr', 'gateway_user': 'cdetry'}, net_devices=set(), _Host__original_extra={'gateway': 'access.grid5000.fr', 'gateway_user': 'cdetry'})}, 'server': {Host(address='gros-86.nancy.grid5000.fr', alias='gros-86.nancy.grid5000.fr', user='root', keyfile=None, port=None, extra={'gateway': 'access.grid5000.fr', 'gateway_user': 'cdetry'}, net_devices=set(), _Host__original_extra={'gateway': 'access.grid5000.fr', 'gateway_user': 'cdetry'})}}

[G5k] gateway is not yet implemented for <class 'enoslib.infra.enos_g5k.objects.G5kEnosProd6Network'> on the G5k side


{'prod': {<enoslib.infra.enos_g5k.objects.G5kEnosProd4Network object at 0x764d2dba78f0>, <enoslib.infra.enos_g5k.objects.G5kEnosProd6Network object at 0x764d2dba7a70>}}

In [19]:
# run the experiment...
print("Launching NPF")
results, _ = npf.run(
    "test_quic.npf",
    series=[
        "local",
        # "local+POWER=2:Quadratic",
    ],  # List of repositories, local means no repository (locally installed software). POWER=2 allows to overwrite variable for a given serie.
    argsv=["--force-retest"],
    roles=roles,
)
df = to_pandas(results)

display(df)

Launching NPF
cluster/gros-83.nancy.grid5000.fr.node could not be found, we will connect to gros-83.nancy.grid5000.fr with SSH using default parameters
cluster/gros-86.nancy.grid5000.fr.node could not be found, we will connect to gros-86.nancy.grid5000.fr with SSH using default parameters
[Local] Running test test_quic.npf...
FCQUIC vs QUIC vs TCP (with TLS)


Executing init scripts...
POISSON = "true", TCP_NODELAY = "true", USE = "false", PER = false [run 1/3 for test 1/1]


/home/corentin/.local/lib/python3.12/site-packages/rich/live.py:221: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Finished 1 tasks ( bash -c 'mkdir -p test2603122309-110101 && cd test2603122309-110101;

echo "Starting server"

env RUST_BACKTRACE=full RUST_LOG="info" RUST_LOG_STYLE=always ./bin/server \
--src  172.16.66.86:4433 --mc-src-addr  172.16.66.86:4443 --test-mode --flexicast \
--fc-timer 0 --fall-back-delay $FALLBACK_DELAY --unicast --fec-scheduler noredundancy > 
server.log  2>&1 &

echo "Server running!"

echo "EVENT server-up"

# SLEEP_LENGTH_SERVER=$(echo "10+${TEST_LENGTH}" | bc)
SLEEP_LENGTH_SERVER=15
echo "Server sleeping for ${SLEEP_LENGTH_SERVER}"
sleep ${SLEEP_LENGTH_SERVER}


') on {'gros-86.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

...

─────────────────────────────────────────────────────────────────────────────────────────────

─────────────────────────────────────────────────────────────────────────────────────────────


server [0] Server running!
server [0] EVENT server-up
server [0] Server sleeping for 15
server [0] Starting server
server [0] Server running!
server [0] Starting server
server [0] Server running!
server [0] EVENT server-up
server [0] Server sleeping for 15
server [0] EVENT server-up
server [0] Server sleeping for 15
Could not find any results ! Something probably went wrong, check the output :
POISSON = "true", TCP_NODELAY = "true", USE = "false", PER = false [run 2/3 for test 1/1]


Program is interrupted

FileNotFoundError: [Errno 2] No such file or directory: 'tmpjyd6eqkz'

In [ ]:
provider.destroy()